In [23]:
import os, json, logging
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Set, TypedDict
from dotenv import load_dotenv

import pandas as pd
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI

logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')
log = logging.getLogger("llm-scheduler")

# Day-pair map
DAYS_MAP: Dict[str, Tuple[str, str]] = {
    "ST": ("Sunday", "Tuesday"),
    "MW": ("Monday", "Wednesday"),
    "RA": ("Thursday", "Saturday"),
}
MAX_UNIQUE_DAYS = 4
GAP_LIMIT_MIN = 120   # minutes

In [25]:
load_dotenv(override=True)

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError("OPENAI_API_KEY not found in .env file. Please check your .env setup.")

In [26]:
llm = ChatOpenAI(
    model="gpt-5-mini",
    temperature=0,
)

In [29]:
# EDIT PATHS if needed
PREADVISE_CSV = "preadvising_191_dept42_PURGED.csv"
SECTIONS_CSV  = "offered_courses_clean_with_times.csv"

preadvise_df = pd.read_csv(PREADVISE_CSV)
sections_df  = pd.read_csv(SECTIONS_CSV)

# --- preadvising: (already filtered to dept 42 / sem 191) ---
preadvise_df.columns = [c.strip() for c in preadvise_df.columns]
if "studentNumber" not in preadvise_df.columns:
    raise ValueError("preadvising CSV must contain 'studentNumber'")
if "courseCode" not in preadvise_df.columns:
    raise ValueError("preadvising CSV must contain 'courseCode'")

# --- offered courses: normalize column names/aliases ---
sections_df.columns = [c.strip() for c in sections_df.columns]

alias_map = {}
# course
if "course" not in sections_df.columns and "courseCode" in sections_df.columns:
    alias_map["courseCode"] = "course"
# section
if "section" not in sections_df.columns and "sec" in sections_df.columns:
    alias_map["sec"] = "section"
# day
if "day" not in sections_df.columns and "dayOfWeek" in sections_df.columns:
    alias_map["dayOfWeek"] = "day"
# start/end
if "start" not in sections_df.columns and "start_time" in sections_df.columns:
    alias_map["start_time"] = "start"
if "end" not in sections_df.columns and "end_time" in sections_df.columns:
    alias_map["end_time"] = "end"

sections_df = sections_df.rename(columns=alias_map)

# seatCapacity -> capacity (fallback to capacity if already present)
cap_source = None
if "seatCapacity" in sections_df.columns:
    cap_source = "seatCapacity"
elif "capacity" in sections_df.columns:
    cap_source = "capacity"
else:
    raise ValueError("offered courses CSV must include 'seatCapacity' or 'capacity'.")

sections_df["capacity"] = pd.to_numeric(sections_df[cap_source], errors="coerce").fillna(0).astype(int)

# final required columns
required = ["course", "section", "day", "start", "end", "capacity"]
missing = [c for c in required if c not in sections_df.columns]
if missing:
    raise ValueError(f"offered courses CSV missing required column(s): {missing}")

# standardize values
sections_df["day"]   = sections_df["day"].astype(str).str.upper().str.strip()
sections_df["start"] = sections_df["start"].astype(str).str.strip()
sections_df["end"]   = sections_df["end"].astype(str).str.strip()

display(preadvise_df.head(3))
display(sections_df.head(3))
print("Capacity > 0 rows:", (sections_df["capacity"] > 0).sum(), "of", len(sections_df))
print("Unique day values sample:", sections_df["day"].unique()[:10])

,id,semesterCode,studentNumber,courseCode,studentDepartment,addedOn,priority
0,706399,191,61781,ENG105,42,2018-12-20 22:50:28,1
1,706400,191,61781,PSY101,42,2018-12-20 22:50:28,1
2,601039,191,82720,CHE101L,42,2018-12-02 18:19:45,1


,semesterCode,departmentCode,course,courseTitle,courseCredit,section,facultyCode,time,room,seatCapacity,course_upper,day,start,end,capacity
0,253,ACT,ACT201,Introduction to Financial Accounting,3.0,1,AhU,RA 11:20 AM - 12:50 PM,NAC510,40,ACT201,RA,11:20,12:50,40
1,253,ACT,ACT201,Introduction to Financial Accounting,3.0,10,IAC,ST 09:40 AM - 11:10 AM,NAC411,40,ACT201,ST,09:40,11:10,40
2,253,ACT,ACT201,Introduction to Financial Accounting,3.0,11,MdM,ST 11:20 AM - 12:50 PM,NAC410,40,ACT201,ST,11:20,12:50,40


Capacity > 0 rows: 1492 of 1640
Unique day values sample: ['RA' 'ST' 'MW' 'R' 'A' 'T' 'W' 'M' 'S' 'F']


In [30]:
def parse_time_to_minutes(t: str) -> int:
    s = str(t).strip().lower().replace(" ", "")
    ampm = None
    if s.endswith("am"):
        ampm = "am"; s = s[:-2]
    elif s.endswith("pm"):
        ampm = "pm"; s = s[:-2]
    if ":" in s:
        h, m = s.split(":", 1)
    else:
        h, m = s, "0"
    h, m = int(h), int(m)
    if ampm == "am" and h == 12: h = 0
    if ampm == "pm" and h != 12: h += 12
    return h*60 + m

@dataclass(frozen=True)
class Section:
    course: str
    section: str
    day_pair: str     # ST / MW / RA
    start_min: int
    end_min: int
    raw_start: str
    raw_end: str
    faculty: Optional[str] = None
    room: Optional[str] = None

    @property
    def days(self) -> Tuple[str, str]:
        return DAYS_MAP[self.day_pair]

def times_overlap(a: Section, b: Section) -> bool:
    if set(a.days).isdisjoint(b.days): return False
    return not (a.end_min <= b.start_min or b.end_min <= a.start_min)

def unique_days(sections: List[Section]) -> int:
    d: Set[str] = set()
    for s in sections: d.update(s.days)
    return len(d)

def daywise_sorted(sections: List[Section]):
    bucket: Dict[str, List[Tuple[int,int,Section]]] = {}
    for s in sections:
        for d in s.days:
            bucket.setdefault(d, []).append((s.start_min, s.end_min, s))
    for k in bucket: bucket[k].sort()
    return bucket

def max_gap_minutes(sections: List[Section]) -> int:
    bucket = daywise_sorted(sections)
    worst = 0
    for _, items in bucket.items():
        for i in range(len(items)-1):
            gap = items[i+1][0] - items[i][1]
            if gap > worst: worst = gap
    return max(0, worst)

In [31]:
class CsvSeatService:
    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()

    def query(self, course: str) -> pd.DataFrame:
        return self.df[self.df["course"].astype(str) == str(course)].copy()

    def rows_to_options(self, rows: pd.DataFrame) -> List[dict]:
        out = []
        for _, r in rows.iterrows():
            dp = str(r["day"]).strip().upper()
            if dp not in DAYS_MAP:   # skip unknown days
                continue
            try:
                s_min = parse_time_to_minutes(r["start"])
                e_min = parse_time_to_minutes(r["end"])
            except Exception:
                continue
            out.append({
                "course": str(r["course"]),
                "section": str(r["section"]),
                "day": dp, "start": str(r["start"]), "end": str(r["end"]),
                "start_min": s_min, "end_min": e_min,
                "capacity": int(r["capacity"]) if pd.notna(r["capacity"]) else 0,
                "faculty": str(r.get("faculty")) if "faculty" in r else None,
                "room": str(r.get("room")) if "room" in r else None,
            })
        return out

    def try_reserve(self, opt: dict, n: int = 1) -> bool:
        m = (
            (self.df["course"].astype(str) == opt["course"]) &
            (self.df["section"].astype(str) == opt["section"]) &
            (self.df["day"].astype(str).str.upper() == opt["day"].upper()) &
            (self.df["start"].astype(str) == opt["start"]) &
            (self.df["end"].astype(str) == opt["end"])
        )
        if m.sum() == 0: return False
        idx = self.df[m].index[0]
        cap = int(self.df.at[idx, "capacity"]) if pd.notna(self.df.at[idx, "capacity"]) else 0
        if cap >= n:
            self.df.at[idx, "capacity"] = cap - n
            return True
        return False

    def release(self, opt: dict, n: int = 1):
        m = (
            (self.df["course"].astype(str) == opt["course"]) &
            (self.df["section"].astype(str) == opt["section"]) &
            (self.df["day"].astype(str).str.upper() == opt["day"].upper()) &
            (self.df["start"].astype(str) == opt["start"]) &
            (self.df["end"].astype(str) == opt["end"])
        )
        if m.sum() == 0: return
        idx = self.df[m].index[0]
        cap = int(self.df.at[idx, "capacity"]) if pd.notna(self.df.at[idx, "capacity"]) else 0
        self.df.at[idx, "capacity"] = cap + n

seat = CsvSeatService(sections_df)

In [32]:
def preferences_by_student(df: pd.DataFrame) -> Dict[str, List[str]]:
    d = {}
    if "priority" in df.columns:
        df2 = df.copy()
        df2["priority"] = pd.to_numeric(df2["priority"], errors="coerce").fillna(9999).astype(int)
        df2 = df2.sort_values(["studentNumber","priority","courseCode"])
    else:
        df2 = df.sort_values(["studentNumber","courseCode"])
    for sid, g in df2.groupby("studentNumber"):
        d[str(sid)] = list(g["courseCode"].astype(str))
    return d

all_prefs = preferences_by_student(preadvise_df)
students = sorted(all_prefs.keys())
print("Students:", len(students))

Students: 1369


In [33]:
def package_supply_for_llm(desired: List[str]):
    """
    Build a compact options list for the LLM. Each option is indexed.
    """
    options = []
    for course in desired:
        rows = seat.query(course)
        opts = seat.rows_to_options(rows)
        for i, opt in enumerate(opts):
            opt["option_id"] = f"{course}__{i}"
            options.append(opt)
    return options

def validate_and_make_sections(plan: Dict[str, str], options: List[dict]) -> Tuple[List[Section], List[str]]:
    """
    plan: {course -> option_id}
    returns (sections, errors). No seat reservations here.
    """
    id_map = {o["option_id"]: o for o in options}
    picked_opts = []
    errors = []
    # assemble
    for course, oid in plan.items():
        if oid not in id_map:
            errors.append(f"Unknown option_id {oid} for {course}")
            continue
        picked_opts.append(id_map[oid])

    # basic checks (no overlaps, ≤4 days)
    secs: List[Section] = []
    for o in picked_opts:
        s = Section(
            course=o["course"], section=o["section"], day_pair=o["day"],
            start_min=o["start_min"], end_min=o["end_min"],
            raw_start=o["start"], raw_end=o["end"], faculty=o.get("faculty"), room=o.get("room")
        )
        # future overlap check happens after we build the list
        secs.append(s)

    # check overlaps
    for i in range(len(secs)):
        for j in range(i+1, len(secs)):
            if times_overlap(secs[i], secs[j]):
                errors.append(f"Time overlap: {secs[i].course}-{secs[i].section} with {secs[j].course}-{secs[j].section}")

    # check days
    if unique_days(secs) > MAX_UNIQUE_DAYS:
        errors.append(f"Unique days {unique_days(secs)} exceed {MAX_UNIQUE_DAYS}")

    return secs, errors

def reserve_plan(secs: List[Section], options: List[dict]) -> Tuple[bool, Optional[str], Optional[dict]]:
    """Try to reserve each; if any fails, return False and the failing option."""
    lookup = {(o["course"], o["section"], o["day"], o["start"], o["end"]): o for o in options}
    reserved = []
    for s in secs:
        key = (s.course, s.section, s.day_pair, s.raw_start, s.raw_end)
        opt = lookup.get(key)
        if not opt:
            return False, "missing_option", None
        ok = seat.try_reserve(opt, 1)
        if not ok:
            # release any prior holds from this attempt
            for ro in reserved:
                seat.release(ro, 1)
            return False, "seat_unavailable", opt
        reserved.append(opt)
    return True, None, None

def release_sections(secs: List[Section]):
    for s in secs:
        seat.release({
            "course": s.course, "section": s.section, "day": s.day_pair,
            "start": s.raw_start, "end": s.raw_end
        }, 1)

In [34]:
class State(TypedDict):
    student: str
    desired: List[str]
    options: List[dict]            # packaged options for LLM
    plan: Dict[str, str]           # {course: option_id}
    schedule: List[Section]        # reserved sections
    feedback: Optional[str]        # "reduce_days" | "reduce_gaps" | None
    drop_list: List[str]           # courses to drop
    message: str
    success: bool
    attempts: int

def adviser_llm(state: State) -> State:
    desired = state.get("desired") or all_prefs.get(state["student"], [])
    if not desired:
        return {**state, "success": False, "message": "No pre-advising rows."}

    # apply drop_list if present
    drops = set(state.get("drop_list") or [])
    desired = [c for c in desired if c not in drops]

    options = package_supply_for_llm(desired)
    # filter out zero-capacity (we still show them, but mark capacity)
    # LLM sees capacity and should avoid zero
    context = {
        "desired": desired,
        "options": [
            {k: o[k] for k in ("option_id","course","section","day","start","end","capacity")}
            for o in options
        ],
        "max_unique_days": MAX_UNIQUE_DAYS,
        "gap_limit_min": GAP_LIMIT_MIN,
        "prev_feedback": state.get("feedback"),
        "note": "Pick exactly ONE option per desired course using option_id. Avoid capacity 0; avoid time overlaps; keep union of days ≤ 4; prefer smaller gaps."
    }

    prompt = (
        "You are an enrollment adviser.\n"
        "Given desired courses and options (with day pair codes ST/MW/RA), choose exactly ONE option_id per course.\n"
        "If it is impossible, propose a drop_list of 1-2 courses to remove (lowest priority first) and return an empty plan.\n"
        "Return strict JSON with keys: plan (object course->option_id), drop_list (array), feedback (string or null).\n"
        f"INPUT JSON:\n{json.dumps(context, ensure_ascii=False)}"
    )
    resp = llm.invoke(prompt).content
    try:
        obj = json.loads(resp)
    except Exception:
        obj = {"plan": {}, "drop_list": [], "feedback": "reduce_days"}

    plan = obj.get("plan") or {}
    drop_list = [str(x) for x in (obj.get("drop_list") or [])]
    feedback = obj.get("feedback")

    # validate plan; if invalid, return to orchestrate again
    secs, errs = validate_and_make_sections(plan, options)
    if errs:
        return {
            **state,
            "desired": desired,
            "options": options,
            "plan": plan,
            "schedule": [],
            "drop_list": drop_list,
            "feedback": feedback or "reduce_days",
            "success": False,
            "message": f"Adviser plan invalid: {errs} (LLM response used)"
        }

    # try to reserve seats now
    ok, cause, failing_opt = reserve_plan(secs, options)
    if not ok:
        # tell the LLM to avoid this option next time by dropping this course or picking another option
        msg = f"Seat reservation failed due to {cause} for {failing_opt}."
        return {
            **state,
            "desired": desired,
            "options": options,
            "plan": plan,
            "schedule": [],
            "drop_list": drop_list,
            "feedback": "reduce_days",   # nudge
            "success": False,
            "message": f"Adviser seat race: {msg}"
        }

    return {
        **state,
        "desired": desired,
        "options": options,
        "plan": plan,
        "schedule": secs,
        "drop_list": drop_list,
        "feedback": None,
        "success": True,
        "message": "Adviser reserved seats for the chosen plan."
    }

def eval_days_llm(state: State) -> State:
    secs = state.get("schedule") or []
    # compute days but let LLM judge/phrase
    data = [{
        "course": s.course, "section": s.section,
        "day_pair": s.day_pair, "days": list(s.days),
        "start": s.raw_start, "end": s.raw_end
    } for s in secs]
    unique = unique_days(secs)
    prompt = (
        "You are a schedule validator for day constraints.\n"
        f"Max unique days allowed: {MAX_UNIQUE_DAYS}.\n"
        "Given the schedule items (with day_pair and real days), determine if it violates the day rule.\n"
        "Return strict JSON: {\"ok\": true/false, \"feedback\": \"reduce_days\" or null}.\n"
        f"INPUT JSON:\n{json.dumps({'schedule': data, 'unique_days': unique}, ensure_ascii=False)}"
    )
    resp = llm.invoke(prompt).content
    try:
        obj = json.loads(resp)
    except Exception:
        obj = {"ok": unique <= MAX_UNIQUE_DAYS, "feedback": None if unique <= MAX_UNIQUE_DAYS else "reduce_days"}

    if not obj.get("ok", False):
        # release holds before replan
        release_sections(secs)
        return {**state, "schedule": [], "feedback": "reduce_days", "success": False, "message": f"Days evaluator: {unique} unique days > {MAX_UNIQUE_DAYS}."}
    return {**state, "feedback": None, "message": f"Days evaluator: OK ({unique} unique days)."}

def eval_gaps_llm(state: State) -> State:
    secs = state.get("schedule") or []
    mg = max_gap_minutes(secs)
    data = [{
        "course": s.course, "section": s.section,
        "day_pair": s.day_pair, "start": s.raw_start, "end": s.raw_end
    } for s in secs]
    prompt = (
        "You are a schedule validator for gaps.\n"
        f"Max gap allowed within a day: {GAP_LIMIT_MIN} minutes.\n"
        "Given the schedule, determine if any day's consecutive classes have gaps exceeding the limit.\n"
        "Return strict JSON: {\"ok\": true/false, \"feedback\": \"reduce_gaps\" or null}.\n"
        f"INPUT JSON:\n{json.dumps({'schedule': data, 'max_gap_found_min': mg}, ensure_ascii=False)}"
    )
    resp = llm.invoke(prompt).content
    try:
        obj = json.loads(resp)
    except Exception:
        obj = {"ok": mg <= GAP_LIMIT_MIN, "feedback": None if mg <= GAP_LIMIT_MIN else "reduce_gaps"}

    if not obj.get("ok", False):
        # release holds before replan
        release_sections(secs)
        return {**state, "schedule": [], "feedback": "reduce_gaps", "success": False, "message": f"Gaps evaluator: max gap {mg} > {GAP_LIMIT_MIN}."}
    return {**state, "feedback": None, "message": f"Gaps evaluator: OK (max gap {mg} min)."}

In [35]:
graph = StateGraph(State)
graph.add_node("adviser", adviser_llm)
graph.add_node("eval_days", eval_days_llm)
graph.add_node("eval_gaps", eval_gaps_llm)

graph.set_entry_point("adviser")

def after_adviser(state: State):
    return "eval_days" if state.get("success") else "adviser"  # loop until adviser returns a seat-reserved plan or drops courses
graph.add_conditional_edges("adviser", after_adviser, {"adviser":"adviser", "eval_days":"eval_days"})

def after_days(state: State):
    return "adviser" if state.get("feedback") == "reduce_days" else "eval_gaps"
graph.add_conditional_edges("eval_days", after_days, {"adviser":"adviser", "eval_gaps":"eval_gaps"})

def after_gaps(state: State):
    return "adviser" if state.get("feedback") == "reduce_gaps" else END
graph.add_conditional_edges("eval_gaps", after_gaps, {"adviser":"adviser", END: END})

app = graph.compile()

In [ ]:
results = []
for i, sid in enumerate(students, 1):
    init: State = {
        "student": sid,
        "desired": [],
        "options": [],
        "plan": {},
        "schedule": [],
        "feedback": None,
        "drop_list": [],
        "message": "",
        "success": False,
        "attempts": 0,
    }
    final = app.invoke(init, config={"recursion_limit": 120})
    sched = final.get("schedule", [])
    results.append({
        "studentNumber": sid,
        "success": bool(final.get("success")),
        "message": final.get("message"),
        "num_sections": len(sched),
        "unique_days": unique_days(sched),
        "max_gap_min": max_gap_minutes(sched),
        "sections": "; ".join(f"{s.course}-{s.section}[{s.day_pair} {s.raw_start}-{s.raw_end}]" for s in sched),
    })
    if i % 25 == 0 or i == len(students):
        print(f"Processed {i}/{len(students)}")

out_df = pd.DataFrame(results)
display(out_df.head(20))
# out_df.to_csv("advising_results_llm_dept42_sem191.csv", index=False)

[INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] HTTP Request: POST https://api.opena

Processed 25/1369


[INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] HTTP Request: POST https://api.opena